In [1]:
PLOT_BOUNDARIES = False

# Data processing


In [2]:
from pathlib import Path
import re

DEBUG = True
N_SUBJECT = 50

FILENAME_PATTERN = re.compile(
    r"(?P<subject_id>\d+)-r(?P<range>\d+)-p(?P<precision>\d+)\.log"
)

figures = Path("paper", "figures")
figures.mkdir(exist_ok=True, parents=True)

## Check failed execution


In [3]:
from typing import Iterable


def get_failed_exec(logs: Path | Iterable[Path]) -> int:
    if isinstance(logs, Path):
        logs = [logs]

    failed = list()
    for log in logs:
        if log.read_text().count("Elapsed time (stage 0):") != 4:
            failed.append(log)
    return failed

In [4]:
# Group logs
import re
from collections import defaultdict


logs = list(Path("logs", "vprec-space_search").rglob("*.log"))

experiements = defaultdict(list)
for log in logs:
    if "binary64" in log.name:
        exp_id = f"{log.parent.name}-binary64"
    else:
        m = FILENAME_PATTERN.match(log.name)
        if not m:
            print(log)
            raise ValueError("Invalid log file name")
        exp_id = f"{log.parent.name}-r{m.group('range')}-p{m.group('precision')}"

    experiements[exp_id].append(log)

In [5]:
# Count failed execution for each range-precision pair
n_failed = defaultdict(dict)
for task_id, logs in experiements.items():
    failed = get_failed_exec(logs)

    if "binary64" in task_id:
        continue
    range_ = int(FILENAME_PATTERN.match(logs[0].name).group("range"))
    precision_ = int(FILENAME_PATTERN.match(logs[0].name).group("precision"))
    n_failed[range_][precision_] = len(failed)

    if len(failed) > 0:
        print(f"Task {task_id} (r{range_}-p{precision_}): {len(failed)} failed")
    if DEBUG and len(failed) != len(logs):
        for log in failed:
            print(log, end="\n\n")

if PLOT_BOUNDARIES:
    # Manually add failure for range 6
    n_failed[6] = {r: N_SUBJECT for r in range(7, 24)}
else:
    del n_failed[8][6]
    del n_failed[7][6]

Task 0011-r7-p6 (r7-p6): 50 failed
Task 0011-r8-p6 (r8-p6): 50 failed
Task 0100-r7-p6 (r7-p6): 50 failed
Task 0100-r8-p6 (r8-p6): 50 failed
Task 0111-r7-p6 (r7-p6): 50 failed
Task 0111-r8-p6 (r8-p6): 50 failed
Task 0110-r7-p6 (r7-p6): 50 failed
Task 0110-r8-p6 (r8-p6): 50 failed
Task 1111-r7-p6 (r7-p6): 50 failed
Task 1111-r8-p6 (r8-p6): 50 failed
Task 0001-r7-p6 (r7-p6): 50 failed
Task 0001-r8-p6 (r8-p6): 50 failed


## QA using metric value


In [6]:
import pandas as pd
import numpy as np


p_lvl_header = r"DIAGNOSTIC,Iteration,metricValue,convergenceValue,ITERATION_TIME_INDEX,SINCE_LAST|  Elapsed time"

dfs = list()

for exp_id, logs in experiements.items():
    for filename in logs:
        # Extract subject_id from log
        subject_id = filename.stem.split("-")[0]

        # Filter out the header from the stages
        txt = filename.read_text()
        all_data = [
            x
            for x in re.split(p_lvl_header, txt)[1:]
            if not x.startswith(" (stage 0):")
        ]

        # 3 stages with 4 levels of resolution each
        for stage in range(1, 4):
            for i, level in enumerate(all_data[4 * (stage - 1) : 4 * stage]):
                table = defaultdict(list)
                for row in re.split(r"\n", level.strip("XX").strip()):
                    # Skip invalid rows
                    # e.g. error messages or write volumes to disk
                    if not ("2DIAGNOSTIC" in row or "1DIAGNOSTIC" in row):
                        continue

                    # Raise exception if the row is not as expected
                    try:
                        cols = row.split(",")
                        table["iterations"].append(cols[1].strip())
                        table["metric"].append(cols[2].strip())
                        table["convergence_value"].append(cols[3].strip())
                        table["total_time"].append(cols[4].strip())
                        table["since_last"].append(cols[5].strip())
                    except Exception as e:
                        print(cols)
                        raise e

                dfs.append(
                    pd.DataFrame(
                        data={
                            "subject": subject_id,
                            "exp_id": exp_id.split("-")[0],
                            "data_format": "-".join(exp_id.split("-")[1:]),
                            "stage": stage,
                            "level": i + 1,
                            "iterations": table["iterations"],
                            "metric": table["metric"],
                            "convergence_value": table["convergence_value"],
                            "total_time": table["total_time"],
                            "since_last": table["since_last"],
                        }
                    )
                )

df = pd.concat(dfs, ignore_index=True)
df = df.astype(
    {
        "stage": int,
        "level": int,
        "iterations": int,
        "metric": np.float64,
        "convergence_value": np.float64,
        "total_time": np.float64,
        "since_last": np.float64,
    }
)
rv = (
    df.groupby(["subject", "exp_id", "data_format", "stage", "level", "iterations"])
    .agg({"metric": "mean", "convergence_value": "mean", "total_time": "mean"})
    .reset_index()
)

In [7]:
rv
rv[
    (rv["data_format"] == "r8-p17")
    & (rv["stage"] == 1)
    & (rv["level"] == 1)
    & (rv["iterations"] == 1)
    & (rv["subject"] == "0003001")
]

,subject,exp_id,data_format,stage,level,iterations,metric,convergence_value,total_time
8847,0003001,0001,r8-p17,1,1,1,-0.570876,1.797693e+308,1.1908
21206,0003001,0011,r8-p17,1,1,1,-0.570876,1.797693e+308,1.2534
34160,0003001,0100,r8-p17,1,1,1,-0.570877,1.797693e+308,0.0000
47301,0003001,0110,r8-p17,1,1,1,-0.570877,1.797693e+308,0.0000
60208,0003001,0111,r8-p17,1,1,1,-0.570877,1.797693e+308,0.0000
70954,0003001,1111,r8-p17,1,1,1,-0.069979,1.797693e+308,0.0000


In [8]:
df
df[
    (df["data_format"] == "r8-p17")
    & (df["stage"] == 1)
    & (df["level"] == 1)
    & (df["iterations"] == 1)
    & (df["subject"] == "0003001")
]


,subject,exp_id,data_format,stage,level,iterations,metric,convergence_value,total_time,since_last
4670,0003001,0011,r8-p17,1,1,1,-0.570876,1.797693e+308,1.2534,1.2534
692675,0003001,0100,r8-p17,1,1,1,-0.570877,1.797693e+308,0.0000,0.0000
1389538,0003001,0111,r8-p17,1,1,1,-0.570877,1.797693e+308,0.0000,0.0000
2040099,0003001,0110,r8-p17,1,1,1,-0.570877,1.797693e+308,0.0000,0.0000
2742043,0003001,1111,r8-p17,1,1,1,-0.069979,1.797693e+308,0.0000,0.0000
3325765,0003001,0001,r8-p17,1,1,1,-0.570876,1.797693e+308,1.1908,1.1908


In [9]:
rv["level"].unique()

array([1, 2, 3, 4])

In [10]:
rv["data_format"].unique()

array(['binary64', 'r7-p10', 'r7-p11', 'r7-p12', 'r7-p13', 'r7-p14',
       'r7-p15', 'r7-p16', 'r7-p17', 'r7-p18', 'r7-p19', 'r7-p20',
       'r7-p21', 'r7-p22', 'r7-p23', 'r7-p6', 'r7-p7', 'r7-p8', 'r7-p9',
       'r8-p10', 'r8-p11', 'r8-p12', 'r8-p13', 'r8-p14', 'r8-p15',
       'r8-p16', 'r8-p17', 'r8-p18', 'r8-p19', 'r8-p20', 'r8-p21',
       'r8-p22', 'r8-p23', 'r8-p6', 'r8-p7', 'r8-p8', 'r8-p9'],
      dtype=object)

In [11]:
# import pandas as pd
# import numpy as np


# p_lvl_header = r"DIAGNOSTIC,Iteration,metricValue,convergenceValue,ITERATION_TIME_INDEX,SINCE_LAST|  Elapsed time"

# dfs = list()
# for array_id in range(1, N_SUBJECT + 1):
#     for filename in logs:
#         if filename.name.startswith("antsRegistration-binary64"):
#             data_format = "binary64"
#         else:
#             data_format = "-".join(filename.name.split("-")[1:3])

#         txt = filename.read_text()

#         # Extract subject_id from log
#         subject_id = re.search(r"SUBJECT_ID: (?P<subject_id>.*)", txt).group(
#             "subject_id"
#         )

#         # Filter out the header from the stages
#         all_data = [
#             x
#             for x in re.split(p_lvl_header, txt)[1:]
#             if not x.startswith(" (stage 0):")
#         ]

#         # 3 stages with 4 levels of resolution each
#         for stage in range(1, 4):
#             for i, level in enumerate(all_data[4 * (stage - 1) : 4 * stage]):
#                 table = defaultdict(list)
#                 for row in re.split(r"\n", level.strip("XX").strip()):
#                     # Skip invalid rows
#                     # e.g. error messages or write volumes to disk
#                     if not ("2DIAGNOSTIC" in row or "1DIAGNOSTIC" in row):
#                         continue

#                     # Raise exception if the row is not as expected
#                     try:
#                         cols = row.split(",")
#                         table["iterations"].append(cols[1].strip())
#                         table["metric"].append(cols[2].strip())
#                         table["convergence_value"].append(cols[3].strip())
#                         table["total_time"].append(cols[4].strip())
#                         table["since_last"].append(cols[5].strip())
#                     except Exception as e:
#                         print(cols)
#                         raise e

#                 dfs.append(
#                     pd.DataFrame(
#                         data={
#                             "subject": subject_id,
#                             "array_id": array_id,
#                             "exp_id": filename.parent.name,
#                             "data_format": data_format,
#                             "stage": stage,
#                             "level": i + 1,
#                             "iterations": table["iterations"],
#                             "metric": table["metric"],
#                             "convergence_value": table["convergence_value"],
#                             "total_time": table["total_time"],
#                             "since_last": table["since_last"],
#                         }
#                     )
#                 )

# df = pd.concat(dfs, ignore_index=True)
# df = df.astype(
#     {
#         "stage": int,
#         "level": int,
#         "iterations": int,
#         "metric": np.float64,
#         "convergence_value": np.float64,
#         "total_time": np.float64,
#         "since_last": np.float64,
#     }
# )
# rv = (
#     df.groupby(
#         ["subject", "array_id", "exp_id", "data_format", "stage", "level", "iterations"]
#     )
#     .agg({"metric": "mean", "convergence_value": "mean", "total_time": "mean"})
#     .reset_index()
# )

In [12]:
def get_stage_metric_stats(df, *, stage=None):
    if stage:
        df_subset = df[(df["stage"] == stage) & (df["level"] == 4)]
    else:
        df_subset = df[df["level"] == 4]
    max_iter_idx = df_subset.groupby(["exp_id", "subject", "data_format"])[
        "iterations"
    ].idxmax()
    df_subset = df_subset.loc[
        max_iter_idx, ["exp_id", "subject", "data_format", "iterations", "metric"]
    ].reset_index(drop=True)
    exp_metrics = (
        df_subset.set_index(["exp_id", "subject", "data_format"])["metric"]
        .unstack()
        .to_dict()
    )

    subjects = df["subject"].unique()
    metric_stats = dict()

    def _get_metric(data, func):
        return {
            k: func(
                [
                    metric
                    for (_, subj_id), metric in v.items()
                    if subj_id in subjects and not np.isnan(metric)
                ]
            )
            for k, v in data.items()
        }

    metric_stats["worst"] = _get_metric(exp_metrics, np.nanmax)
    metric_stats["mean"] = _get_metric(exp_metrics, np.nanmean)
    metric_stats["best"] = _get_metric(exp_metrics, np.nanmin)
    metric_stats["raw_value"] = _get_metric(exp_metrics, lambda x: x)

    return metric_stats

# Figures


In [31]:
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import plotly.graph_objects as go


def _highlight_cell(fig, *, x, y):
    fig.add_shape(
        type="rect",
        x0=x - 0.5,
        y0=y - 0.5,
        x1=x + 0.5,
        y1=y + 0.5,
        line=dict(color="black", width=2),
        xref="x",
        yref="y",
    )


def plot_metric(
    stage_metrics: dict[str, dict[float]],
    max_abs_diff: float,
    *,
    metric_func: Callable,
    std_delta_func: Callable,
    title: str,
    out_file: Optional[Path] = None,
):
    # Extract x, y, and intensity values
    data_formats = set(stage_metrics["mean"].keys()) - {"binary64"}
    x = sorted(
        {
            int(data_format.split("-")[1].removeprefix("p"))
            for data_format in data_formats
        }
    )
    y = sorted(
        {
            int(data_format.split("-")[0].removeprefix("r"))
            for data_format in data_formats
        }
    )
    z = np.zeros((len(y), len(x)))

    # Populate the intensity grid
    for yi, y_key in enumerate(y):
        for xi, x_key in enumerate(x):
            z[yi, xi] = stage_metrics["mean"].get(f"r{y_key}-p{x_key}", np.nan)

    # Compute metric transformation
    delta = metric_func(z, stage_metrics["mean"])

    # Define custom colorscale
    colorscale = [
        [0, "green"],
        [0.5, "white"],
        [1, "red"],
    ]
    print(
        ",".join(
            [
                f"max_abs_diff: {max_abs_diff:.4f}",
            ]
        )
    )

    # Convert NaN to 1 to display with colorscale
    masked_failed = np.where(np.isnan(delta), 1, np.nan)
    delta = np.where(np.isnan(delta), np.nan, delta)

    # Set format for delta values
    std_delta = std_delta_func(x, y, stage_metrics["raw_value"])
    formatter = np.vectorize(lambda d, s: f"{d:.2e}<br>± {s:.2e}")
    formatted = formatter(delta, std_delta)

    # Set text labels for heatmap
    text_labels = np.where(delta <= 1, formatted, "")

    # Create heatmap
    normal_trace = go.Heatmap(
        z=delta,
        x=x,
        y=y,
        colorscale=colorscale,
        showscale=False,
        zmin=-max_abs_diff,
        zmax=max_abs_diff,
        zmid=0,
        text=text_labels,
        texttemplate="%{text}",
    )
    failed_trace = go.Heatmap(
        z=masked_failed,
        x=x,
        y=y,
        colorscale=[[0, "grey"], [1, "grey"]],
        showscale=False,
    )
    fig = go.Figure(data=[failed_trace, normal_trace])

    # Annotate hardware data format
    _highlight_cell(fig, x=23, y=8)  # Float32
    _highlight_cell(fig, x=16, y=7)  # FP24
    _highlight_cell(fig, x=10, y=8)  # TensorFloat32
    _highlight_cell(fig, x=7, y=8)  # BFloat16

    # Style
    fig.update_layout(
        title=title,
        xaxis=dict(
            title="Precision (bits)",
            tickvals=x,
            ticks="",
            title_font=dict(size=24),
        ),
        yaxis=dict(
            title="Range (bits)",
            tickvals=y,
            ticks="",
            title_font=dict(size=24),
        ),
        width=1600,
        height=200,
        margin=dict(l=10, r=10, t=40, b=10),
    )

    fig.show()

    # Save figure
    if out_file:
        fig.update_layout(
            title=None,
            margin=dict(l=10, r=10, t=10, b=100),
            width=2000,
            height=250,
            font=dict(
                size=24,
                color="black",
            ),
        )
        out_file.parent.mkdir(exist_ok=True, parents=True)
        fig.write_image(out_file)
        # fig.write_image(out_file.with_suffix(".png"))

In [32]:
import warnings

warnings.filterwarnings("ignore")

STAGES = [("rigid", "MI"), ("affine", "MI"), ("syn", "CC")]
EXP_IDS = [
    "0111",
    "0110",
    "0100",
    "0011",
    "0001",
    # "1111",
]


def rel_diff(z, metrics):
    return (z - metrics["binary64"]) / np.abs(metrics["binary64"])


def rel_diff_std(x, y, exp_metrics):
    std_delta = np.zeros((len(y), len(x)))
    for yi, y_key in enumerate(y):
        for xi, x_key in enumerate(x):
            std_delta[yi, xi] = np.nanstd(
                np.abs(
                    np.asarray(exp_metrics.get(f"r{y_key}-p{x_key}", np.nan))
                    - np.asarray(exp_metrics["binary64"])
                )
                / np.abs(exp_metrics["binary64"])
            )
    return std_delta


def get_experiment(df, *, exp_id=[]):
    # Always include the binary64 data format
    if isinstance(exp_id, str):
        exp_id = [exp_id]
    return df[df["exp_id"].isin(exp_id + ["0000"])]


def get_stage_max_abs_diff(df: pd.DataFrame, stage: int):
    mean_metrics = get_stage_metric_stats(get_experiment(rv, exp_id=EXP_IDS), stage=3)[
        "mean"
    ]

    raw_metric = np.array(
        [
            x
            for y in get_stage_metric_stats(
                get_experiment(rv, exp_id=EXP_IDS), stage=3
            )["raw_value"].values()
            for x in y
        ]
    )

    return np.nanmax(rel_diff(raw_metric, mean_metrics))


# Compute on a per-stage basis
all_data = get_experiment(rv, exp_id=EXP_IDS)
global_max_abs_diff = np.max(
    [get_stage_max_abs_diff(all_data, stage=stage) for stage in [1, 2, 3]]
)

for exp_id in EXP_IDS:
    exp_data = get_experiment(rv, exp_id=exp_id)
    n_subjects = exp_data["subject"].nunique()

    for stage_id, (stage_name, metric_name) in enumerate(STAGES, 1):
        # Skip unecessary plots
        if (stage_name == "rigid" and exp_id != "0111") or (
            stage_name == "affine" and exp_id in ["0110", "0001"]
        ):
            continue

        stage_metric_stats = get_stage_metric_stats(
            get_experiment(rv, exp_id=EXP_IDS), stage=stage_id
        )
        exp_metric_stats = get_stage_metric_stats(exp_data, stage=stage_id)

        """Metric relative difference with binary64"""
        plot_metric(
            exp_metric_stats,
            max_abs_diff=global_max_abs_diff,
            title=f"[{exp_id}] {stage_name.capitalize()} ({metric_name}): Relative difference with Binary64 - Subjects mean",
            out_file=figures.joinpath(
                f"{exp_id}-{stage_name}-metric_rel_diff-mean.pdf"
            ),
            metric_func=rel_diff,
            std_delta_func=rel_diff_std,
        )

max_abs_diff: 0.1197


max_abs_diff: 0.1197


max_abs_diff: 0.1197


max_abs_diff: 0.1197


max_abs_diff: 0.1197


max_abs_diff: 0.1197


max_abs_diff: 0.1197


max_abs_diff: 0.1197


max_abs_diff: 0.1197
